In [1]:
import os
import sys
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..')) # Adjust as needed
if project_root not in sys.path:
    sys.path.append(project_root) # add notebook to sys.path

In [2]:
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"torch.accelerater.current_accelerator() gives {device} device")

torch.accelerater.current_accelerator() gives mps device


# Dataset 1: Time Machine Character prediction

In [4]:
from utils.data import TimeMachineDataset
from utils.models import RNNLM
from utils.losses import loss_fn_tm

In [5]:
data = TimeMachineDataset(num_steps=9, batch_size=4, num_train=10000, num_val=5000, device=device)
train_dl, val_dl, vocab = data.get_data()

In [6]:
model = RNNLM(len(vocab), hidden_dim=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

In [7]:
losses = []

for X, Y in train_dl:
    logits = model(X)
    loss = loss_fn_tm(logits, Y)
    loss.backward()
    opt.step()
    opt.zero_grad()
    losses.append(loss.item())

In [8]:
print(losses[0], losses[-1])

3.3719866275787354 2.051948308944702


# Dataset #2: Machine Translation French English

In [9]:
from utils.data import FraEngMTDataset
from utils.models import Seq2Seq, Seq2SeqEncoder, Seq2SeqAttentionDecoder
from utils.losses import loss_fn_mt

In [10]:
mt_dataset = FraEngMTDataset(batch_size=4, num_steps=9, num_train=512, num_val=128, device=device)
train_dl, val_dl, src_vocab, tgt_vocab = mt_dataset.get_data()
encoder = Seq2SeqEncoder(len(src_vocab), embed_size=32, hidden_size=32, num_layers=2, dropout=0.2)
decoder = Seq2SeqAttentionDecoder(len(tgt_vocab), embed_size=32, num_hiddens=32, num_layers=2, dropout=0.2)
model = Seq2Seq(encoder, decoder, tgt_vocab['<pad>']).to(device)

In [11]:
opt = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = lambda Y_hat, Y: loss_fn_mt(model, Y_hat, Y)

In [12]:
for src, tgt, src_valid_len, tgt_label in train_dl:
    print(src.shape, tgt.shape, src_valid_len.shape, tgt_label.shape)
    print(src.device, tgt.device, src_valid_len.device, tgt_label.device)
    break

torch.Size([4, 9]) torch.Size([4, 9]) torch.Size([4]) torch.Size([4, 9])
mps:0 mps:0 mps:0 mps:0


In [13]:
losses = []

for src, tgt, src_valid_len, tgt_label in train_dl:
    logits = model(src, tgt, src_valid_len)
    loss = loss_fn(logits, tgt_label)
    loss.backward()
    opt.step()
    opt.zero_grad()
    losses.append(loss.item())

In [14]:
print(losses[0], losses[-1])

5.3815813064575195 3.0186519622802734
